# Deep Learning Baseline for Sen1Floods11 in Google Colab

**UPDATED Apr 21 20:25**

This Colab version now includes:

- direct Sen1Floods11 download from public GCS
- weighted cross entropy
- full-data-ready training loop with early stopping
- threshold sweep with CSV export
- experiment summary CSV export

Look for the cell named: `# Threshold sweep for water probability and experiment CSV export`.


In [ ]:
# Colab environment setup
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

# Colab already includes PyTorch on GPU runtimes. We install only the geospatial / utility packages
# that are often missing or version-sensitive.
%pip -q install rasterio tqdm scikit-learn matplotlib pandas numpy


In [ ]:
# Imports and reproducibility
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


In [ ]:
# Optional: download the Sen1Floods11 workflow data directly inside Colab
# Run this cell if you do NOT already have sen1floods11_workflow in Drive.
# It downloads 500 weakly-labeled training chips and the official hand-labeled validation split.

import csv
import json
import shutil
import subprocess
import urllib.parse
import urllib.request

DOWNLOAD_DATA = True
DOWNLOAD_ROOT = Path("/content/sen1floods11_workflow")
N_WEAK_TRAIN = 500
GCS_BASE = "gs://sen1floods11/v1.1"
HTTP_BASE = "https://storage.googleapis.com/sen1floods11/v1.1"


def gs_to_http(gs_url):
    if not gs_url.startswith(GCS_BASE):
        raise ValueError(f"Unexpected GCS url: {gs_url}")
    return gs_url.replace(GCS_BASE, HTTP_BASE, 1)


def download_public_file(url, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if not out_path.exists():
        print("Downloading", url)
        urllib.request.urlretrieve(url, out_path)
    return out_path


def gsutil_cp_many(urls, dest_dir):
    urls = list(urls)
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    if not urls:
        return
    print(f"Downloading {len(urls)} files to {dest_dir}")
    result = subprocess.run(
        ["gsutil", "-m", "cp", "-n", "-I", str(dest_dir)],
        input="\n".join(urls) + "\n",
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout[-2000:])
    if result.returncode != 0:
        print("gsutil had some failures; missing files will be retried with HTTPS fallback.")
        if result.stderr:
            print(result.stderr[-2000:])


def ensure_downloaded(url_path_pairs):
    """Verify local files after batch download; retry missing files one by one via HTTPS."""
    missing = [(url, Path(path)) for url, path in url_path_pairs if not Path(path).exists()]
    if not missing:
        return

    print(f"Retrying {len(missing)} missing files with HTTPS fallback")
    for gs_url, out_path in missing:
        download_public_file(gs_to_http(gs_url), out_path)

    still_missing = [str(path) for _, path in missing if not Path(path).exists()]
    if still_missing:
        raise FileNotFoundError(f"Still missing files after retry, examples: {still_missing[:5]}")


def copy_alias(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not src.exists():
        raise FileNotFoundError(f"Downloaded source file is missing: {src}")
    if not dst.exists():
        shutil.copy2(src, dst)


def list_first_weak_s1(n):
    # Public Google Cloud Storage JSON API; no auth needed.
    api_url = "https://storage.googleapis.com/storage/v1/b/sen1floods11/o"
    params = urllib.parse.urlencode({
        "prefix": "v1.1/data/flood_events/WeaklyLabeled/S1Weak/",
        "maxResults": n,
    })
    with urllib.request.urlopen(f"{api_url}?{params}") as response:
        payload = json.loads(response.read().decode("utf-8"))
    names = [item["name"] for item in payload.get("items", [])]
    names = [name for name in names if name.endswith("_S1Weak.tif")]
    return names[:n]


def prepare_sen1floods11_workflow(root=DOWNLOAD_ROOT, n_weak=N_WEAK_TRAIN):
    root = Path(root)
    manifest_path = root / "manifest.csv"
    if manifest_path.exists():
        print("Existing manifest found:", manifest_path)
        return root

    print("Preparing data in", root)
    (root / "splits").mkdir(parents=True, exist_ok=True)

    weak_s1_names = list_first_weak_s1(n_weak)
    if len(weak_s1_names) < n_weak:
        raise RuntimeError(f"Only found {len(weak_s1_names)} weak S1 files; expected {n_weak}.")

    weak_records = []
    weak_s1_pairs = []
    weak_label_pairs = []
    for object_name in weak_s1_names:
        s1_name = Path(object_name).name
        chip_id = s1_name.removesuffix("_S1Weak.tif")
        label_name = f"{chip_id}_S2IndexLabelWeak.tif"

        s1_url = f"{GCS_BASE}/data/flood_events/WeaklyLabeled/S1Weak/{s1_name}"
        label_url = f"{GCS_BASE}/data/flood_events/WeaklyLabeled/S2IndexLabelWeak/{label_name}"
        s1_raw = root / "weak_train_s2index" / "S1Weak" / s1_name
        label_raw = root / "weak_train_s2index" / "S2IndexLabelWeak" / label_name

        weak_s1_pairs.append((s1_url, s1_raw))
        weak_label_pairs.append((label_url, label_raw))
        weak_records.append((chip_id, s1_name, label_name))

    gsutil_cp_many([url for url, _ in weak_s1_pairs], root / "weak_train_s2index" / "S1Weak")
    ensure_downloaded(weak_s1_pairs)
    gsutil_cp_many([url for url, _ in weak_label_pairs], root / "weak_train_s2index" / "S2IndexLabelWeak")
    ensure_downloaded(weak_label_pairs)

    rows = []
    weak_split_csv = root / "splits" / f"weak_train_s2index_{n_weak}.csv"
    with weak_split_csv.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["chip_id", "s1weak_original", "s2index_label_original", "s1_alias", "qc_alias"])
        for chip_id, s1_name, label_name in weak_records:
            s1_raw = root / "weak_train_s2index" / "S1Weak" / s1_name
            label_raw = root / "weak_train_s2index" / "S2IndexLabelWeak" / label_name
            s1_alias = root / "weak_train_s2index" / "S1" / f"{chip_id}_S1.tif"
            label_alias = root / "weak_train_s2index" / "QC" / f"{chip_id}_QC.tif"
            copy_alias(s1_raw, s1_alias)
            copy_alias(label_raw, label_alias)
            writer.writerow([
                chip_id,
                s1_name,
                label_name,
                str(s1_alias.relative_to(root)),
                str(label_alias.relative_to(root)),
            ])
            rows.append({
                "split": "weak_train_s2index",
                "label_source": "S2IndexLabelWeak",
                "chip_id": chip_id,
                "s1_path": str(s1_alias.relative_to(root)),
                "label_path": str(label_alias.relative_to(root)),
                "original_s1": str(s1_raw.relative_to(root)),
                "original_label": str(label_raw.relative_to(root)),
            })

    valid_csv = download_public_file(
        f"{HTTP_BASE}/splits/flood_handlabeled/flood_valid_data.csv",
        root / "splits" / "flood_valid_data.csv",
    )
    hand_records = []
    with valid_csv.open(newline="") as f:
        for s1_name, label_name in csv.reader(f):
            chip_id = s1_name.removesuffix("_S1Hand.tif")
            hand_records.append((chip_id, s1_name, label_name))

    hand_s1_pairs = []
    hand_label_pairs = []
    for _, s1_name, label_name in hand_records:
        hand_s1_pairs.append((
            f"{GCS_BASE}/data/flood_events/HandLabeled/S1Hand/{s1_name}",
            root / "hand_validation" / "S1Hand" / s1_name,
        ))
        hand_label_pairs.append((
            f"{GCS_BASE}/data/flood_events/HandLabeled/LabelHand/{label_name}",
            root / "hand_validation" / "LabelHand" / label_name,
        ))

    gsutil_cp_many([url for url, _ in hand_s1_pairs], root / "hand_validation" / "S1Hand")
    ensure_downloaded(hand_s1_pairs)
    gsutil_cp_many([url for url, _ in hand_label_pairs], root / "hand_validation" / "LabelHand")
    ensure_downloaded(hand_label_pairs)

    for chip_id, s1_name, label_name in hand_records:
        s1_raw = root / "hand_validation" / "S1Hand" / s1_name
        label_raw = root / "hand_validation" / "LabelHand" / label_name
        s1_alias = root / "hand_validation" / "S1" / f"{chip_id}_S1.tif"
        label_alias = root / "hand_validation" / "QC" / f"{chip_id}_QC.tif"
        copy_alias(s1_raw, s1_alias)
        copy_alias(label_raw, label_alias)
        rows.append({
            "split": "hand_validation",
            "label_source": "LabelHand",
            "chip_id": chip_id,
            "s1_path": str(s1_alias.relative_to(root)),
            "label_path": str(label_alias.relative_to(root)),
            "original_s1": str(s1_raw.relative_to(root)),
            "original_label": str(label_raw.relative_to(root)),
        })

    pd.DataFrame(rows).to_csv(manifest_path, index=False)
    print("Wrote", manifest_path)
    print(pd.DataFrame(rows)["split"].value_counts())
    return root


if DOWNLOAD_DATA:
    DATA_ROOT = prepare_sen1floods11_workflow()
else:
    DATA_ROOT = None


In [ ]:
# Data and output paths
# If the download cell ran, DATA_ROOT is already set to /content/sen1floods11_workflow.
# If auto-detection does not find your data, edit DATA_ROOT manually.
try:
    DATA_ROOT
except NameError:
    DATA_ROOT = None

candidate_roots = [
    Path("/content/sen1floods11_workflow"),
    Path("/content/data/sen1floods11_workflow"),
    Path("/content/drive/MyDrive/musa650-final/data/sen1floods11_workflow"),
    Path("/content/drive/MyDrive/musa650-final/sen1floods11_workflow"),
    Path("/content/drive/MyDrive/Colab Notebooks/sen1floods11_workflow"),
    Path("/content/drive/MyDrive/sen1floods11_workflow"),
]

if DATA_ROOT is None:
    for root in candidate_roots:
        if (root / "manifest.csv").exists():
            DATA_ROOT = root
            break

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not find sen1floods11_workflow/manifest.csv. "
        "Run the download cell above, or set DATA_ROOT manually in this cell."
    )

DATA_ROOT = Path(DATA_ROOT)
OUTPUT_DIR = Path("/content/drive/MyDrive/musa650-final/outputs") if Path("/content/drive/MyDrive").exists() else Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# Read manifest and build train/validation splits
manifest = pd.read_csv(DATA_ROOT / "manifest.csv")
print(manifest.head())
print(manifest.columns.tolist())

if {"split", "s1_path", "label_path"}.issubset(manifest.columns):
    train_df = manifest[manifest["split"].str.contains("train", case=False, na=False)].copy()
    val_df = manifest[manifest["split"].str.contains("validation|val", case=False, na=False)].copy()
else:
    # Also supports the smaller sample manifest: s1_alias / qc_alias with train and validation_hand paths.
    manifest = manifest.rename(columns={"s1_alias": "s1_path", "qc_alias": "label_path"})
    train_df = manifest[manifest["s1_path"].str.contains("train", case=False, na=False)].copy()
    val_df = manifest[manifest["s1_path"].str.contains("validation|val", case=False, na=False)].copy()

for df in (train_df, val_df):
    df["s1_abs"] = df["s1_path"].apply(lambda p: DATA_ROOT / p)
    df["label_abs"] = df["label_path"].apply(lambda p: DATA_ROOT / p)

missing = [p for p in list(train_df["s1_abs"].head(3)) + list(val_df["label_abs"].head(3)) if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f"Some manifest paths do not exist. First missing examples: {missing[:3]}")

print(f"Training chips: {len(train_df):,}")
print(f"Validation chips: {len(val_df):,}")


In [ ]:
# Dataset utilities
IGNORE_INDEX = -1
NUM_CLASSES = 2


def normalize_s1(x):
    """Robust per-chip normalization for two-band Sentinel-1 arrays shaped C x H x W."""
    x = x.astype("float32")
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    out = np.empty_like(x, dtype="float32")
    for b in range(x.shape[0]):
        band = x[b]
        lo, hi = np.percentile(band, [2, 98])
        if hi <= lo:
            out[b] = 0.0
        else:
            out[b] = np.clip((band - lo) / (hi - lo), 0, 1)
            out[b] = out[b] * 2 - 1
    return out


class Sen1FloodsDataset(Dataset):
    def __init__(self, frame, augment=False):
        self.frame = frame.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with rasterio.open(row.s1_abs) as src:
            x = src.read([1, 2])
        with rasterio.open(row.label_abs) as src:
            y = src.read(1).astype("int64")

        x = normalize_s1(x)
        y = np.where(np.isin(y, [0, 1]), y, IGNORE_INDEX).astype("int64")

        if self.augment:
            if random.random() < 0.5:
                x = x[:, :, ::-1].copy()
                y = y[:, ::-1].copy()
            if random.random() < 0.5:
                x = x[:, ::-1, :].copy()
                y = y[::-1, :].copy()

        return torch.from_numpy(x), torch.from_numpy(y)


train_ds = Sen1FloodsDataset(train_df, augment=True)
val_ds = Sen1FloodsDataset(val_df, augment=False)

x0, y0 = train_ds[0]
print("Input:", tuple(x0.shape), x0.dtype, "Label:", tuple(y0.shape), y0.dtype, "Label values:", torch.unique(y0))


In [ ]:
# Class balance and weighted cross entropy setup
CLASS_WEIGHT_MODE = "auto"  # "auto", "balanced", or "manual"
MANUAL_CLASS_WEIGHTS = [1.0, 8.0]  # [non_water, water], only used when mode == "manual"


def count_pixels(frame):
    counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    for label_path in tqdm(frame["label_abs"], desc="Counting class pixels"):
        with rasterio.open(label_path) as src:
            y = src.read(1)
        counts[0] += int(np.sum(y == 0))
        counts[1] += int(np.sum(y == 1))
    return counts


def make_class_weights(counts, mode="auto", manual=(1.0, 8.0)):
    if mode == "manual":
        weights = np.array(manual, dtype="float32")
    else:
        freq = counts / counts.sum()
        inv = 1.0 / np.maximum(freq, 1e-8)
        weights = np.sqrt(inv) if mode == "auto" else inv
        weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)

class_counts = count_pixels(train_df)
class_freq = class_counts / class_counts.sum()
class_weights = make_class_weights(class_counts, CLASS_WEIGHT_MODE, MANUAL_CLASS_WEIGHTS).to(device)

print("Train valid pixel counts [non_water, water]:", class_counts.tolist())
print("Train class frequency [non_water, water]:", class_freq.round(6).tolist())
print("CE weights [non_water, water]:", class_weights.detach().cpu().numpy().round(4).tolist())


In [ ]:
# DataLoaders
BATCH_SIZE = 4
NUM_WORKERS = 2

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


In [ ]:
# Model: selectable U-Net variants
# Keep MODEL_VARIANT = "small_unet" for the original baseline.
# Try MODEL_VARIANT = "res_attention_unet" when class-weight/LR tuning stops improving.
MODEL_VARIANT = "res_attention_unet"  # "small_unet" or "res_attention_unet"
BASE_CHANNELS = 32
DROPOUT = 0.10


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet(nn.Module):
    def __init__(self, in_channels=2, num_classes=2, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 4, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)


class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        self.shortcut = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv2d(in_channels, out_channels, 1, bias=False)
        )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class AttentionGate(nn.Module):
    def __init__(self, gating_channels, skip_channels, inter_channels):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Conv2d(gating_channels, inter_channels, 1, bias=False),
            nn.BatchNorm2d(inter_channels),
        )
        self.skip = nn.Sequential(
            nn.Conv2d(skip_channels, inter_channels, 1, bias=False),
            nn.BatchNorm2d(inter_channels),
        )
        self.psi = nn.Sequential(
            nn.Conv2d(inter_channels, 1, 1),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, gating, skip):
        attention = self.psi(self.relu(self.gate(gating) + self.skip(skip)))
        return skip * attention


class AttentionUpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.attn = AttentionGate(out_channels, skip_channels, max(out_channels // 2, 1))
        self.conv = ResidualConvBlock(out_channels + skip_channels, out_channels, dropout=dropout)

    def forward(self, x, skip):
        x = self.up(x)
        skip = self.attn(x, skip)
        return self.conv(torch.cat([x, skip], dim=1))


class ResidualAttentionUNet(nn.Module):
    """A stronger U-Net: residual encoder/decoder blocks plus attention-gated skip connections."""
    def __init__(self, in_channels=2, num_classes=2, base=32, dropout=0.10):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.enc1 = ResidualConvBlock(in_channels, base, dropout=0.0)
        self.enc2 = ResidualConvBlock(base, base * 2, dropout=dropout)
        self.enc3 = ResidualConvBlock(base * 2, base * 4, dropout=dropout)
        self.enc4 = ResidualConvBlock(base * 4, base * 8, dropout=dropout)
        self.bottleneck = ResidualConvBlock(base * 8, base * 16, dropout=dropout)
        self.up4 = AttentionUpBlock(base * 16, base * 8, base * 8, dropout=dropout)
        self.up3 = AttentionUpBlock(base * 8, base * 4, base * 4, dropout=dropout)
        self.up2 = AttentionUpBlock(base * 4, base * 2, base * 2, dropout=dropout)
        self.up1 = AttentionUpBlock(base * 2, base, base, dropout=0.0)
        self.head = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.up4(b, e4)
        d3 = self.up3(d4, e3)
        d2 = self.up2(d3, e2)
        d1 = self.up1(d2, e1)
        return self.head(d1)


def build_model(variant=MODEL_VARIANT, base=BASE_CHANNELS, dropout=DROPOUT):
    if variant == "small_unet":
        return SmallUNet(in_channels=2, num_classes=2, base=base)
    if variant == "res_attention_unet":
        return ResidualAttentionUNet(in_channels=2, num_classes=2, base=base, dropout=dropout)
    raise ValueError(f"Unknown MODEL_VARIANT: {variant}")


model = build_model().to(device)
print("MODEL_VARIANT:", MODEL_VARIANT)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")


In [ ]:
# Metrics
@torch.no_grad()
def segmentation_metrics(logits, target):
    pred = logits.argmax(dim=1)
    valid = target != IGNORE_INDEX
    if valid.sum() == 0:
        return {
            "accuracy": np.nan,
            "iou_water": np.nan,
            "f1_water": np.nan,
            "true_water_frac": np.nan,
            "pred_water_frac": np.nan,
        }

    pred_v = pred[valid]
    target_v = target[valid]

    accuracy = (pred_v == target_v).float().mean().item()
    tp = ((pred_v == 1) & (target_v == 1)).sum().item()
    fp = ((pred_v == 1) & (target_v == 0)).sum().item()
    fn = ((pred_v == 0) & (target_v == 1)).sum().item()

    iou = tp / (tp + fp + fn + 1e-8)
    f1 = (2 * tp) / (2 * tp + fp + fn + 1e-8)
    true_water_frac = (target_v == 1).float().mean().item()
    pred_water_frac = (pred_v == 1).float().mean().item()
    return {
        "accuracy": accuracy,
        "iou_water": iou,
        "f1_water": f1,
        "true_water_frac": true_water_frac,
        "pred_water_frac": pred_water_frac,
    }


def average_dicts(items):
    out = {}
    for key in items[0]:
        vals = np.array([item[key] for item in items], dtype="float64")
        out[key] = float(np.nanmean(vals))
    return out


In [ ]:
# Training loop
EPOCHS = 80
LR = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 20

USE_DICE_LOSS = True
DICE_LOSS_WEIGHT = 0.25

ce_criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=IGNORE_INDEX)

def dice_loss_water(logits, target, ignore_index=-1, eps=1e-6):
    probs = torch.softmax(logits, dim=1)[:, 1]
    valid = target != ignore_index

    if valid.sum() == 0:
        return logits.sum() * 0.0

    target_water = (target == 1).float()
    probs = probs[valid]
    target_water = target_water[valid]

    intersection = (probs * target_water).sum()
    denominator = probs.sum() + target_water.sum()
    dice = (2 * intersection + eps) / (denominator + eps)
    return 1 - dice

def combined_loss(logits, target):
    ce_loss = ce_criterion(logits, target)
    if USE_DICE_LOSS:
        d_loss = dice_loss_water(logits, target, ignore_index=IGNORE_INDEX)
        return ce_loss + DICE_LOSS_WEIGHT * d_loss
    return ce_loss

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

best_iou = -1
epochs_without_improvement = 0
history = []
best_model_path = OUTPUT_DIR / "sen1floods11_small_unet_best.pt"

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    model.train()
    train_losses = []
    train_metrics = []

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} train"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            logits = model(x)
            loss = combined_loss(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_losses.append(loss.item())
        train_metrics.append(segmentation_metrics(logits.detach(), y))

    scheduler.step()

    model.eval()
    val_losses = []
    val_metrics = []
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} val"):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            val_losses.append(combined_loss(logits, y).item())
            val_metrics.append(segmentation_metrics(logits, y))

    train_metric_avg = average_dicts(train_metrics)
    metrics = average_dicts(val_metrics)
    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        "val_loss": float(np.mean(val_losses)),
        **metrics,
        "train_pred_water_frac": train_metric_avg["pred_water_frac"],
        "lr": optimizer.param_groups[0]["lr"],
        "seconds": time.time() - start,
    }
    history.append(row)

    improved = row["iou_water"] > best_iou
    if improved:
        best_iou = row["iou_water"]
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "metrics": row,
            "class_counts": class_counts.tolist(),
            "class_weights": class_weights.detach().cpu().tolist(),
            "data_root": str(DATA_ROOT),
            "model_variant": MODEL_VARIANT,
            "base_channels": BASE_CHANNELS,
            "dropout": DROPOUT,
            "use_dice_loss": USE_DICE_LOSS,
            "dice_loss_weight": DICE_LOSS_WEIGHT,
        }, best_model_path)
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:02d}: train_loss={row['train_loss']:.4f} "
        f"val_loss={row['val_loss']:.4f} acc={row['accuracy']:.3f} "
        f"water_iou={row['iou_water']:.3f} water_f1={row['f1_water']:.3f} "
        f"true_water={row['true_water_frac']:.3f} pred_water={row['pred_water_frac']:.3f}"
    )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping after {epoch} epochs; best validation water IoU={best_iou:.3f}.")
        break

history_df = pd.DataFrame(history)
history_path = OUTPUT_DIR / "sen1floods11_small_unet_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved best model to:", best_model_path)
print("Saved training history to:", history_path)



In [ ]:
# Plot training curves
history_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["iou_water"], label="Water IoU")
axes[1].plot(history_df["epoch"], history_df["f1_water"], label="Water F1")
axes[1].set_title("Validation metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(history_df["epoch"], history_df["true_water_frac"], label="true water")
axes[2].plot(history_df["epoch"], history_df["pred_water_frac"], label="pred water")
axes[2].set_title("Collapse check")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.tight_layout()


In [ ]:
# Threshold sweep for water probability and experiment CSV export
# This caches validation probabilities once, then scans thresholds quickly.
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_probs = []
all_targets = []
losses = []

with torch.no_grad():
    for x, y in tqdm(val_loader, desc="cache validation probabilities"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        losses.append(combined_loss(logits, y).item())
        prob_water = torch.softmax(logits, dim=1)[:, 1].detach().cpu()
        all_probs.append(prob_water.reshape(-1))
        all_targets.append(y.detach().cpu().reshape(-1))

probs = torch.cat(all_probs)
targets = torch.cat(all_targets)
valid = targets != IGNORE_INDEX
probs = probs[valid]
targets = targets[valid]

thresholds = np.linspace(0.05, 0.95, 19)
stats = []
for threshold in thresholds:
    pred = (probs >= threshold).long()
    tp = int(((pred == 1) & (targets == 1)).sum().item())
    fp = int(((pred == 1) & (targets == 0)).sum().item())
    fn = int(((pred == 0) & (targets == 1)).sum().item())
    tn = int(((pred == 0) & (targets == 0)).sum().item())
    iou = tp / (tp + fp + fn + 1e-8)
    f1 = 2 * tp / (2 * tp + fp + fn + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    stats.append({
        "experiment": "colab_full_weighted_ce_dice_threshold_sweep" if USE_DICE_LOSS else "colab_full_weighted_ce_threshold_sweep",
        "threshold": float(threshold),
        "val_loss": float(np.mean(losses)),
        "iou_water": iou,
        "f1_water": f1,
        "precision": precision,
        "recall": recall,
        "true_water_frac": float((targets == 1).float().mean().item()),
        "pred_water_frac": float((pred == 1).float().mean().item()),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "checkpoint_epoch": checkpoint.get("epoch"),
        "model_variant": MODEL_VARIANT,
        "base_channels": BASE_CHANNELS,
        "use_dice_loss": USE_DICE_LOSS,
        "dice_loss_weight": DICE_LOSS_WEIGHT if USE_DICE_LOSS else 0.0,
        "class_weight_0": float(class_weights.detach().cpu()[0]),
        "class_weight_1": float(class_weights.detach().cpu()[1]),
    })

threshold_df = pd.DataFrame(stats)
threshold_path = OUTPUT_DIR / "sen1floods11_threshold_sweep.csv"
threshold_df.to_csv(threshold_path, index=False)
print("Saved threshold sweep:", threshold_path)
display(threshold_df.sort_values("iou_water", ascending=False).head(10))

best_threshold = threshold_df.loc[threshold_df["iou_water"].idxmax()]
BEST_WATER_THRESHOLD = float(best_threshold["threshold"])

history_for_summary = pd.DataFrame(history)
best_history = history_for_summary.loc[history_for_summary["iou_water"].idxmax()]
experiment_summary = pd.DataFrame([
    {
        "experiment": "colab_full_weighted_ce_dice_batch_avg_argmax" if USE_DICE_LOSS else "colab_full_weighted_ce_batch_avg_argmax",
        "epochs_run": int(history_for_summary["epoch"].max()),
        "best_epoch": int(best_history["epoch"]),
        "decision_rule": "argmax / batch-averaged metrics",
        "threshold": 0.50,
        "val_iou_water": float(best_history["iou_water"]),
        "val_f1_water": float(best_history["f1_water"]),
        "val_accuracy": float(best_history["accuracy"]),
        "val_true_water_frac": float(best_history["true_water_frac"]),
        "val_pred_water_frac": float(best_history["pred_water_frac"]),
        "use_dice_loss": USE_DICE_LOSS,
        "dice_loss_weight": DICE_LOSS_WEIGHT if USE_DICE_LOSS else 0.0,
        "notes": "Training metric; averages IoU/F1 per batch.",
    },
    {
        "experiment": "colab_full_weighted_ce_dice_global_threshold" if USE_DICE_LOSS else "colab_full_weighted_ce_global_threshold",
        "epochs_run": int(history_for_summary["epoch"].max()),
        "best_epoch": int(checkpoint.get("epoch", best_history["epoch"])),
        "decision_rule": "softmax water threshold / global confusion",
        "threshold": BEST_WATER_THRESHOLD,
        "val_iou_water": float(best_threshold["iou_water"]),
        "val_f1_water": float(best_threshold["f1_water"]),
        "val_accuracy": np.nan,
        "val_true_water_frac": float(best_threshold["true_water_frac"]),
        "val_pred_water_frac": float(best_threshold["pred_water_frac"]),
        "use_dice_loss": USE_DICE_LOSS,
        "dice_loss_weight": DICE_LOSS_WEIGHT if USE_DICE_LOSS else 0.0,
        "notes": "Recommended validation readout; computes confusion over all valid pixels.",
    },
])
summary_path = OUTPUT_DIR / "sen1floods11_experiment_summary.csv"
experiment_summary.to_csv(summary_path, index=False)
print("Saved experiment summary:", summary_path)
display(experiment_summary)
print("Recommended BEST_WATER_THRESHOLD:", BEST_WATER_THRESHOLD)



In [ ]:
# Visualize predictions on validation chips
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

n_show = min(4, len(val_ds))
fig, axes = plt.subplots(n_show, 4, figsize=(14, 3.5 * n_show))
if n_show == 1:
    axes = axes[None, :]

for i in range(n_show):
    x, y = val_ds[i]
    with torch.no_grad():
        logits = model(x.unsqueeze(0).to(device))
        threshold = globals().get("BEST_WATER_THRESHOLD", 0.50)
        prob_water = torch.softmax(logits, dim=1)[:, 1]
        pred = (prob_water >= threshold).squeeze(0).cpu().numpy().astype("uint8")

    vv = x[0].numpy()
    vh = x[1].numpy()
    label = y.numpy()
    label_plot = np.ma.masked_where(label == IGNORE_INDEX, label)

    axes[i, 0].imshow(vv, cmap="gray")
    axes[i, 0].set_title("VV normalized")
    axes[i, 1].imshow(vh, cmap="gray")
    axes[i, 1].set_title("VH normalized")
    axes[i, 2].imshow(label_plot, cmap="Blues", vmin=0, vmax=1)
    axes[i, 2].set_title("Label")
    axes[i, 3].imshow(pred, cmap="Blues", vmin=0, vmax=1)
    axes[i, 3].set_title("Prediction")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()


In [ ]:
# Optional: run inference on one chip and save a predicted mask as GeoTIFF
# Change sample_row to point at a different row if needed.
sample_row = val_df.iloc[0]
out_tif = OUTPUT_DIR / f"{sample_row.get('chip_id', 'sample')}_pred_water.tif"

with rasterio.open(sample_row.s1_abs) as src:
    x_np = normalize_s1(src.read([1, 2]))
    profile = src.profile.copy()

with torch.no_grad():
    x_tensor = torch.from_numpy(x_np).unsqueeze(0).to(device)
    pred = model(x_tensor).argmax(dim=1).squeeze(0).cpu().numpy().astype("uint8")

profile.update(count=1, dtype="uint8", nodata=255)
with rasterio.open(out_tif, "w", **profile) as dst:
    dst.write(pred, 1)

print("Saved prediction GeoTIFF:", out_tif)
